# Start here

This is the code companion to the lecture note *Energy System Models*. The
note explains the economics and never mentions code; these notebooks are
where you meet the models that produced its figures.

**Nothing in the note requires you to run any of this.** If you would rather
read and think, close the laptop — you will not be at a disadvantage. These
notebooks exist for the things that are genuinely easier to learn by turning
a knob: what happens to the price when you raise demand, how a value factor
collapses, why a peaker gets built.

## What you need

Python 3.11+ and the packages in `environment.yml` at the repository root.
If the cell below runs, you are set up.

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
note = next(p for p in [here, *here.parents] if (p / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

print(f"note root: {note}")

In [ ]:
import pypsa, linopy
print("pypsa  ", pypsa.__version__)
print("linopy ", linopy.__version__)
print("\nIf that printed two version numbers, everything you need is installed.")

## The smallest model in the note

Section 2 of the note is a single-hour dispatch problem: ten technologies,
one demand level, and the question of which plants run. Here it is.

First the technology table — the same file the models read, and the same
numbers printed as Table 2.1 in the note.

In [ ]:
from model import dispatch

tech = dispatch.read_tech(PROCESSED / "technology_costs_small.csv")
tech[["label", "fuel", "efficiency", "fuel_price_eur_per_mwh_th",
      "vom_eur_per_mwh", "co2_t_per_mwh_th"]]

Marginal cost and emission rate are *derived* from that table, not assumed:

$$ c_g = \frac{p^F_g}{\eta_g} + o_g, \qquad e_g = \frac{\varphi_g}{\eta_g} $$

In [ ]:
pd.DataFrame({
    "marginal cost (EUR/MWh)": dispatch.marginal_cost(tech).round(1),
    "emission rate (t/MWh)":   dispatch.emission_rate(tech).round(3),
}).sort_values("marginal cost (EUR/MWh)")

## Solving it, and reading the economics off the duals

This is the whole point of the note in one cell. We solve a cost-minimisation
and get back not just the dispatch but **the price** and **every plant's
rent** — because those are the dual variables of the constraints.

In [ ]:
from pipeline.run_dispatch import CAPACITY_MW, LOAD_MW

sol = dispatch.solve(tech, CAPACITY_MW, load=LOAD_MW)

print(f"load            {LOAD_MW:,.0f} MW")
print(f"price (lambda)  {sol['price']:.2f} EUR/MWh   <- dual of market clearing")
print(f"system cost     {sol['cost']:,.0f} EUR")
print(f"emissions       {sol['emissions']:,.0f} t")

In [ ]:
pd.DataFrame({
    "capacity (MW)":   CAPACITY_MW,
    "dispatch (MW)":   sol["generation"].round(0),
    "marginal cost":   sol["marginal_cost"].round(1),
    "scarcity rent":   sol["rent"].round(1),
}).sort_values("marginal cost")

Read that table against Proposition 1 in the note:

- every plant with $c_g < \lambda$ runs at capacity and earns a rent
  $\mu_g = \lambda - c_g$;
- every plant with $c_g > \lambda$ is idle and earns nothing;
- exactly one plant is *marginal*: partly dispatched, earning zero rent, and
  setting the price.

Find the marginal plant in the table above. Its marginal cost should equal
the price printed in the previous cell.

## Your turn

1. Raise `LOAD_MW` until the oil peaker sets the price. What is the price then?
2. At what load does the model become infeasible, and why? (Section 2.4 of the
   note is about exactly this.)
3. Add a carbon price: `dispatch.solve(tech, CAPACITY_MW, LOAD_MW, co2_price=85)`.
   Which plant leaves the dispatch?

In [ ]:
# Try it here.

## Where to go next

| Notebook | Note section |
|---|---|
| `01-dispatch-and-the-price.ipynb` | §2 — merit order, duality, carbon tax vs cap |
| `02-intermittency.ipynb` | §3 — weather, capture prices, cannibalisation |
| `03-storage.ipynb` | §4 — arbitrage, ramping, what storage does to the price |
| `04-heat.ipynb` | §5 — a heat pump's moving COP; two prices, one exchange rate |
| `05-transmission.ipynb` | §6 — zonal prices, congestion rent, a bigger cable |
| `06-investment.ipynb` | §7 — screening curves, what Denmark should build, a CO2 budget |
| `07-expansion.ipynb` | §8 — every zone extendable, one budget, the ladder |
| `08-uncertainty.ipynb` | §9 — weather years, cost scenarios, tax versus cap |
| `09-reproducing-the-figures.ipynb` | how every figure in the note is made |

The assistant in this repository can help: ask it, or use `/model-explainer`
to have a piece of model code explained economically.